In [6]:
!pip -q install transformers datasets sentence-transformers
import time
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

In [7]:
ds = load_dataset("fancyzhx/ag_news")
news = ds["train"].shuffle(seed=7).select(range(5000)).to_pandas()
texts = news["text"].tolist()

print("Number of articles:", len(texts))
print(texts[0])

Number of articles: 5000
Air France-KLM Sales Rise 6.4 on Passenger Increase (Update1) Air France-KLM Group, Europe #39;s biggest airline, said second-quarter sales grew 6.4 percent as passengers generated more revenue.


In [8]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    pipe_device = 0
else:
    device = "cpu"
    pipe_device = -1

print("Using device:", device)

embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

start = time.time()
embeddings = embedder.encode(texts, batch_size=128, convert_to_tensor=True, show_progress_bar=True)
embed_time = time.time() - start

print("Embedding shape:", embeddings.shape)
print("Time to embed 5000 articles (seconds):", round(embed_time, 1))

Using device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Embedding shape: torch.Size([5000, 384])
Time to embed 5000 articles (seconds): 254.3


In [9]:
def search(query, top_k=3):
    query_embedding = embedder.encode(query, convert_to_tensor=True)
    hits = util.semantic_search(query_embedding, embeddings, top_k=top_k)[0]
    for h in hits:
        print("Score:", round(h["score"], 3))
        print(texts[h["corpus_id"]])
        print("-----")

search("Which company released a new phone?")

Score: 0.568
PalmOne rings up new smartphone PalmOne Inc., the world #39;s biggest maker of hand-held computers, unveiled a new model of its popular Treo smartphone, the combined mobile handset, camera and planner that sold more than 600,000 units in a year.
-----
Score: 0.533
Gaming Cell Phone: Take Two Nokia #39;s N-Gage QD addresses many of its predecessor #39;s problems, but it #39;s still a big phone. Nokia #39;s N-Gage gaming cell phone was the Gigli of the tech world when it first came out last year.
-----
Score: 0.489
Nokia Raises Outlook on Global Phone Sales By MATTI HUUHTANEN    HELSINKI, Finland (AP) -- Nokia said Thursday its third-quarter revenue and profits would be better than expected, citing an increase in mobile phone sales. The news pushed its shares up nearly 6.5 percent in trading...
-----


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

qa_name = "distilbert-base-cased-distilled-squad"
qa_tok = AutoTokenizer.from_pretrained(qa_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_name).to(device)
qa_model.eval()

def answer_question(question, context):
    inputs = qa_tok(question, context, return_tensors="pt",
                    truncation="only_second", max_length=384).to(device)
    with torch.no_grad():
        outputs = qa_model(**inputs)

    start_probs = outputs.start_logits.softmax(-1)[0]
    end_probs = outputs.end_logits.softmax(-1)[0]

    start_idx = int(start_probs.argmax())
    end_idx = int(end_probs.argmax())

    if end_idx < start_idx:
        return "", 0.0

    score = float(start_probs[start_idx] * end_probs[end_idx])
    answer_ids = inputs["input_ids"][0][start_idx:end_idx + 1]
    answer = qa_tok.decode(answer_ids, skip_special_tokens=True)
    return answer, score

def ask(question):
    query_embedding = embedder.encode(question, convert_to_tensor=True)
    hits = util.semantic_search(query_embedding, embeddings, top_k=5)[0]

    best_answer = ""
    best_score = 0
    best_text = ""

    for h in hits:
        article = texts[h["corpus_id"]]
        answer, score = answer_question(question, article)
        if score > best_score:
            best_score = score
            best_answer = answer
            best_text = article

    print("QUESTION:", question)
    print("ANSWER:", best_answer)
    print("CONFIDENCE:", round(best_score, 2))
    print("FOUND IN:", best_text)
    print("-----")

ask("Who won the match?")
ask("What did the company announce?")
ask("Where did the attack happen?")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

QUESTION: Who won the match?
ANSWER: Amelie Mauresmo
CONFIDENCE: 0.7
FOUND IN: Mauresmo wins first match as world No. 1 Amelie Mauresmo overcame a slow start Thursday to win her first match as the world #39;s top ranked player, beating Patty Schnyder of Switzerland 7-5, 6-4 at the Porsche Grand Prix.
-----
QUESTION: What did the company announce?
ANSWER: application infrastructure software company
CONFIDENCE: 0.26
FOUND IN: BEA Systems to Host Financial Analyst Conference on December 9 &lt;b&gt;...&lt;/b&gt; SAN JOSE, Calif., Dec. 8 /PRNewswire-FirstCall/ -- (http://bea.com/more_info.jsp?p8 amp;r=1.4.6 ) -- BEA Systems, Inc. (NASDAQ:BEAS), the world #39;s leading application infrastructure software company, today announced 
-----
QUESTION: Where did the attack happen?
ANSWER: southern \ Afghanistan
CONFIDENCE: 0.83
FOUND IN: Blast Wounds Three U.S. Soldiers, Afghan Interpreter (Reuters) Reuters - A roadside bomb exploded in southern\Afghanistan, wounding three U.S. soldiers and an Afgh